In [1]:
import pandas as pd
import numpy as np
import warnings
from sklearn.exceptions import UndefinedMetricWarning
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
import joblib
import os

# Tắt cảnh báo chia cho 0 do dữ liệu mất cân bằng
warnings.filterwarnings('ignore', category=UndefinedMetricWarning)

# ==========================================
# 1. ĐỌC DỮ LIỆU & CHUẨN BỊ LỚP
# ==========================================
df = pd.read_csv('/kaggle/input/datasets/yasserh/wine-quality-dataset/WineQT.csv')

print(" Phân bố điểm chất lượng gốc (quality):")
print(df['quality'].value_counts().sort_index())

def map_quality_group(q):
    if q <= 4: return 0
    elif q <= 6: return 1
    else: return 2

df['quality_group'] = df['quality'].apply(map_quality_group)
print("\n Phân bố lớp sau khi gộp (0: Kém | 1: Trung bình | 2: Tốt):")
print(df['quality_group'].value_counts().sort_index())

# ==========================================
# 2. CHIA DỮ LIỆU & CHUẨN HÓA
# ==========================================
X = df.drop(columns=['Id', 'quality', 'quality_group'])
y = df['quality_group']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("\n Đã chuẩn hóa dữ liệu thành công.")

os.makedirs('/kaggle/working', exist_ok=True)

# ==========================================
# 3. HUẤN LUYỆN & ĐÁNH GIÁ RIÊNG TỪNG MÔ HÌNH
# ==========================================

# ------------------------------------------
# 3.1 LOGISTIC REGRESSION
# ------------------------------------------
print("\n" + "="*50)
print(" MÔ HÌNH 1: LOGISTIC REGRESSION")
print("="*50)

lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)
acc_lr = accuracy_score(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr, average='weighted', zero_division=0)

print(f" Accuracy : {acc_lr:.4f}")
print(f" F1-Score : {f1_lr:.4f}")
print("\n Classification Report:")
print(classification_report(y_test, y_pred_lr, target_names=['Kém', 'Trung bình', 'Tốt'], zero_division=0))

joblib.dump(lr, '/kaggle/working/wine_quality_logreg.pkl')
print(" Đã lưu: /kaggle/working/wine_quality_logreg.pkl")

# ------------------------------------------
# 3.2 RANDOM FOREST
# ------------------------------------------
print("\n" + "="*50)
print(" MÔ HÌNH 2: RANDOM FOREST")
print("="*50)

rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf.fit(X_train_scaled, y_train)

y_pred_rf = rf.predict(X_test_scaled)
acc_rf = accuracy_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf, average='weighted', zero_division=0)

print(f" Accuracy : {acc_rf:.4f}")
print(f" F1-Score : {f1_rf:.4f}")
print("\n Classification Report:")
print(classification_report(y_test, y_pred_rf, target_names=['Kém', 'Trung bình', 'Tốt'], zero_division=0))

joblib.dump(rf, '/kaggle/working/wine_quality_rf.pkl')
print("💾 Đã lưu: /kaggle/working/wine_quality_rf.pkl")

# ------------------------------------------
# 3.3 KNN (TÌM K TỐI ƯU -> HUẤN LUYỆN)
# ------------------------------------------
print("\n" + "="*50)
print(" BƯỚC TỐI ƯU: TÌM K TỐT NHẤT CHO KNN")
print("="*50)

best_k = 5
best_acc_knn = 0
for k in range(3, 15):
    knn_temp = KNeighborsClassifier(n_neighbors=k, weights='distance')
    knn_temp.fit(X_train_scaled, y_train)
    acc_temp = knn_temp.score(X_test_scaled, y_test)
    print(f"  k={k} | Accuracy: {acc_temp:.4f}")
    if acc_temp > best_acc_knn:
        best_acc_knn = acc_temp
        best_k = k

print(f" Chọn k = {best_k} cho KNN (Accuracy: {best_acc_knn:.4f})")

print("\n" + "="*50)
print(" MÔ HÌNH 3: KNN")
print("="*50)

knn = KNeighborsClassifier(n_neighbors=best_k, weights='distance', metric='euclidean')
knn.fit(X_train_scaled, y_train)

y_pred_knn = knn.predict(X_test_scaled)
acc_knn = accuracy_score(y_test, y_pred_knn)
f1_knn = f1_score(y_test, y_pred_knn, average='weighted', zero_division=0)

print(f" Accuracy : {acc_knn:.4f}")
print(f" F1-Score : {f1_knn:.4f}")
print("\n Classification Report:")
print(classification_report(y_test, y_pred_knn, target_names=['Kém', 'Trung bình', 'Tốt'], zero_division=0))

joblib.dump(knn, '/kaggle/working/wine_quality_knn.pkl')
print(" Đã lưu: /kaggle/working/wine_quality_knn.pkl")

# ==========================================
#  4. LỰA CHỌN & LƯU MÔ HÌNH TỐT NHẤT
# ==========================================
print("\n" + "="*50)
print(" TỔNG HỢP & CHỌN MODEL TỐT NHẤT")
print("="*50)

# Gom kết quả vào dictionary để dễ so sánh
models_results = {
    "Logistic Regression": {"model": lr, "acc": acc_lr, "f1": f1_lr},
    "Random Forest": {"model": rf, "acc": acc_rf, "f1": f1_rf},
    f"KNN (k={best_k})": {"model": knn, "acc": acc_knn, "f1": f1_knn}
}

# Chọn model có F1-Score cao nhất (ưu tiên F1 vì dữ liệu mất cân bằng)
best_model_name = max(models_results, key=lambda x: models_results[x]["f1"])
best_model = models_results[best_model_name]["model"]

# Lưu model tốt nhất
joblib.dump(best_model, '/kaggle/working/wine_quality_best_model.pkl')
print(f" Mô hình tốt nhất theo F1-Score: {best_model_name}")
print(" Đã lưu file thứ 4: /kaggle/working/wine_quality_best_model.pkl")

# ==========================================
# 5. LƯU SCALER & TỔNG KẾT
# ==========================================
joblib.dump(scaler, '/kaggle/working/scaler.pkl')
print("\n Đã lưu scaler: /kaggle/working/scaler.pkl")

print("\n" + "="*65)
print(" BẢNG TỔNG KẾT KẾT QUẢ (4 FILES ĐÃ LƯU)")
print("="*65)
print(f"{'Mô hình':<25} | {'Accuracy':<10} | {'F1-Score':<10} | {'Đánh giá'}")
print("-" * 70)
for name, res in models_results.items():
    badge = " TỐT NHẤT" if name == best_model_name else ""
    print(f"{name:<25} | {res['acc']:<10.4f} | {res['f1']:<10.4f} | {badge}")

print("\n Danh sách file đã lưu tại /kaggle/working/:")
print("  1. wine_quality_logreg.pkl")
print("  2. wine_quality_rf.pkl")
print("  3. wine_quality_knn.pkl")
print("  4. wine_quality_best_model.pkl ")
print("  5. scaler.pkl")

 Phân bố điểm chất lượng gốc (quality):
quality
3      6
4     33
5    483
6    462
7    143
8     16
Name: count, dtype: int64

 Phân bố lớp sau khi gộp (0: Kém | 1: Trung bình | 2: Tốt):
quality_group
0     39
1    945
2    159
Name: count, dtype: int64

 Đã chuẩn hóa dữ liệu thành công.

 MÔ HÌNH 1: LOGISTIC REGRESSION
 Accuracy : 0.5895
 F1-Score : 0.6581

 Classification Report:
              precision    recall  f1-score   support

         Kém       0.08      0.62      0.15         8
  Trung bình       0.94      0.55      0.69       189
         Tốt       0.45      0.81      0.58        32

    accuracy                           0.59       229
   macro avg       0.49      0.66      0.47       229
weighted avg       0.84      0.59      0.66       229

 Đã lưu: /kaggle/working/wine_quality_logreg.pkl

 MÔ HÌNH 2: RANDOM FOREST
 Accuracy : 0.8996
 F1-Score : 0.8810

 Classification Report:
              precision    recall  f1-score   support

         Kém       0.00      0.00     